# 00. Download NOAA OISST v2.1 Monthly SST

Download the monthly NOAA OISST v2.1 dataset used by the workflow.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "manuscript.yml").exists() and (candidate / "src" / "eams_seof").exists():
            return candidate
    raise FileNotFoundError("Repository root not found.")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from eams_seof.config import load_config, resolve_path

CONFIG_PATH = REPO_ROOT / "configs" / "manuscript.yml"
cfg = load_config(CONFIG_PATH, repo_root=REPO_ROOT)

print("Repository:", REPO_ROOT)
print("Config:", CONFIG_PATH)


In [ ]:
import subprocess
import numpy as np
import xarray as xr
from netCDF4 import Dataset, num2date, date2num

raw_path = resolve_path(REPO_ROOT, cfg["paths"]["oisst_raw"])
raw_path.parent.mkdir(parents=True, exist_ok=True)

download_url = (
    "https://downloads.psl.noaa.gov/Datasets/"
    "noaa.oisst.v2.highres/sst.mon.mean.nc"
)

opendap_url = (
    "https://psl.noaa.gov/thredds/dodsC/"
    "Datasets/noaa.oisst.v2.highres/sst.mon.mean.nc"
)


def download_full_file():
    """Download the complete monthly OISST file (first run only)."""
    command = ["wget", "-c", download_url, "-O", str(raw_path)]
    print(" ".join(command))
    subprocess.run(command, check=True)
    print("Downloaded full file:", raw_path)


def update_existing_file():
    """Append only remote timesteps that are newer than the local file."""

    # ---------------------------------------------------------
    # 1. Read the local time axis
    # ---------------------------------------------------------
    with Dataset(raw_path, "r") as nc:
        if "time" not in nc.variables:
            raise RuntimeError("Local file has no 'time' variable.")

        if not nc.dimensions["time"].isunlimited():
            raise RuntimeError(
                "Local 'time' dimension is not unlimited; cannot append safely."
            )

        local_time_var = nc.variables["time"]

        if len(local_time_var) == 0:
            raise RuntimeError("Local file has an empty time axis.")

        local_time_units = local_time_var.units
        local_calendar = getattr(local_time_var, "calendar", "standard")
        local_last_num = local_time_var[-1].item()

        local_last_date = num2date(
            local_last_num,
            units=local_time_units,
            calendar=local_calendar,
            only_use_cftime_datetimes=True,
        )

        local_lat = np.asarray(nc.variables["lat"][:])
        local_lon = np.asarray(nc.variables["lon"][:])

    print("Local latest timestamp :", local_last_date)

    # ---------------------------------------------------------
    # 2. Read only the remote coordinates/time axis first
    # ---------------------------------------------------------
    with xr.open_dataset(
        opendap_url,
        engine="netcdf4",
        decode_times=False,
        mask_and_scale=False,
    ) as remote:

        remote_time = remote["time"]
        remote_time_values = remote_time.load().values

        remote_time_units = remote_time.attrs["units"]
        remote_calendar = remote_time.attrs.get("calendar", "standard")

        remote_dates = num2date(
            remote_time_values,
            units=remote_time_units,
            calendar=remote_calendar,
            only_use_cftime_datetimes=True,
        )

        new_idx = np.array(
            [i for i, date in enumerate(remote_dates) if date > local_last_date],
            dtype=int,
        )

        remote_latest_date = remote_dates[-1]
        print("Remote latest timestamp:", remote_latest_date)

        if new_idx.size == 0:
            print("OISST is already up to date.")
            return

        # OISST monthly time is monotonic, so new records should be contiguous.
        i0 = int(new_idx[0])

        if not np.array_equal(new_idx, np.arange(i0, len(remote_time_values))):
            raise RuntimeError(
                "Unexpected non-contiguous new timesteps in the remote dataset."
            )

        # Basic grid consistency check before modifying the local file.
        remote_lat = remote["lat"].load().values
        remote_lon = remote["lon"].load().values

        if not np.array_equal(local_lat, remote_lat):
            raise RuntimeError("Remote latitude grid differs from the local file.")

        if not np.array_equal(local_lon, remote_lon):
            raise RuntimeError("Remote longitude grid differs from the local file.")

        print(
            f"Downloading {len(new_idx)} new timestep(s): "
            f"{remote_dates[i0]} -> {remote_dates[-1]}"
        )

        # Only the missing SST fields are transferred here.
        new_sst = (
            remote["sst"]
            .isel(time=slice(i0, None))
            .load()
            .values
        )

        new_dates = remote_dates[i0:]

    # ---------------------------------------------------------
    # 3. Convert remote dates to the local time convention
    # ---------------------------------------------------------
    new_local_time = date2num(
        new_dates,
        units=local_time_units,
        calendar=local_calendar,
    )

    # ---------------------------------------------------------
    # 4. Append to the local NetCDF file
    # ---------------------------------------------------------
    with Dataset(raw_path, "a") as nc:
        n0 = len(nc.dimensions["time"])
        nnew = len(new_local_time)

        nc.variables["time"][n0:n0 + nnew] = new_local_time
        nc.variables["sst"][n0:n0 + nnew, :, :] = new_sst

        nc.sync()

    print(f"Appended {nnew} timestep(s).")
    print("Updated:", raw_path)


# -------------------------------------------------------------
# First run: full download
# Later runs: incremental update only
# -------------------------------------------------------------
if not raw_path.exists():
    download_full_file()
else:
    try:
        update_existing_file()
    except OSError as exc:
        raise RuntimeError(
            f"Existing file could not be opened as NetCDF: {raw_path}\n"
            "If this is an interrupted/corrupted first download, remove the "
            "partial file and run this cell again."
        ) from exc

## Inspect the source dataset

Check the downloaded dataset and its metadata.

In [ ]:
import xarray as xr

ds = xr.open_dataset(raw_path)
display(ds)

print("First timestamp :", ds.time.values[0])
print("Latest timestamp:", ds.time.values[-1])
print("Variables       :", list(ds.data_vars))
